# 12 — Real-Time AML Simulation Dashboard

Interactive Dash dashboard with **Start/Stop/Reset** controls and live-updating Plotly charts.

```
Main Thread (Dash @ 8052)             Background Thread (simulation)
├─ dcc.Interval (500ms)                ├─ stream.next_batch()
├─ callbacks read SimulationState       ├─ graph.update()
├─ Start/Stop/Reset buttons             ├─ scorer.score()
├─ render Plotly figures                 └─ write to SimulationState (Lock)
└─ 3 tabs: Control, Analysis, Investigation
```

| Tab | Purpose |
|-----|--------|
| **Control & Monitor** | Config panel, KPIs, live score chart, alert feed, confusion matrix, score histogram |
| **Analysis** | Post-simulation summary, latency analysis |
| **Investigation** | Ego-graph of top alert, node details |

### Prerequisites
Run notebooks **01–05** first to generate model artifacts in `models/` and `data/`.

In [ ]:
# ══ Imports & GPU Safety ══
import os, sys, json, time, gc, math, threading
from datetime import datetime, timedelta
from collections import deque, defaultdict
from copy import deepcopy

import numpy as np
import pandas as pd
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

import dash
from dash import dcc, html, dash_table, Input, Output, State, ctx
from dash.dash_table.Format import Format, Scheme
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px

from gan_anomaly import Generator, Encoder, anomaly_score

# GPU cleanup from any previous notebook
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.memory_allocated()/1e6:.1f} MB allocated")

# ── Pipeline integration ──────────────────────────────────────────
_RUN_DIR = os.environ.get("AML_RUN_DIR", "")
DATA_DIR = os.path.join(_RUN_DIR, "data") if _RUN_DIR else "data"
MODELS_DIR = os.path.join(_RUN_DIR, "models") if _RUN_DIR else "models"

print(f"RUN_DIR: {_RUN_DIR or '(local)'}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"MODELS_DIR: {MODELS_DIR}")

In [2]:
# ═══════════════════════════════════════════════════════════
#  CONFIGURATION
# ═══════════════════════════════════════════════════════════

DEFAULT_CONFIG = {
    "NUM_BATCHES": 80,
    "BATCH_SIZE": 50,
    "BATCH_INTERVAL": 0.5,
    "PATTERN_PROB": 0.10,
    "RESCORE_INTERVAL": 3,
    "THRESHOLD_PERCENTILE": 99,
    "ALERT_HISTORY": 200,
    "SCORE_HISTORY": 200,
    "SEED": 42,
}

# Transaction generation constants (same as NB 00/11)
AMOUNT_MU = 6.2
AMOUNT_SIGMA = 0.4
AMOUNT_MIN = 40.0
AMOUNT_MAX = 3_000.0
TX_TYPES = ["TRANSFER-Mutual", "TRANSFER-FanOut", "TRANSFER-Forward", "TRANSFER-Periodical", "TRANSFER-FanIn"]
TX_PROBS = np.array([0.31, 0.26, 0.22, 0.19, 0.02])
TX_PROBS /= TX_PROBS.sum()

FAN_MIN_LEGS = 4
FAN_MAX_LEGS = 8
PATTERN_AMT_MIN = 500
PATTERN_AMT_MAX = 3_000
STRUCTURING_NUM_TXNS = 8
STRUCTURING_AMT_MIN = 7_000
STRUCTURING_AMT_MAX = 9_500

print(f"Default: {DEFAULT_CONFIG['NUM_BATCHES']} batches \u00d7 {DEFAULT_CONFIG['BATCH_SIZE']} txns")
print(f"All parameters adjustable via dashboard controls")

Default: 80 batches × 50 txns
All parameters adjustable via dashboard controls


In [3]:
# ═══════════════════════════════════════════════════════════
#  GraphSAGE Model (from NB 02/11)
# ═══════════════════════════════════════════════════════════

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    @torch.no_grad()
    def full_forward(self, x, edge_index):
        self.eval()
        return self.forward(x, edge_index)

print("GraphSAGE class defined (SAGEConv 9\u2192128\u219264)")

GraphSAGE class defined (SAGEConv 9→128→64)


In [4]:
# ═══════════════════════════════════════════════════════════
#  TransactionStream (from NB 11)
# ═══════════════════════════════════════════════════════════

class TransactionStream:
    """Simulates a live transaction feed with optional laundering patterns."""

    PATTERN_TYPES = ["fan_out", "fan_in", "cycle", "scatter_gather", "structuring"]

    def __init__(self, party_ids, rng, pattern_prob=0.10):
        self.party_ids = list(party_ids)
        self.n_parties = len(self.party_ids)
        self.rng = rng
        self.pattern_prob = pattern_prob
        self.tran_counter = 1_000_000
        self.alert_counter = 500
        self.base_time = datetime.now()
        self.batch_num = 0
        self.total_patterns_injected = 0

    def _pick(self, n, exclude=None):
        pool = self.party_ids if exclude is None else [p for p in self.party_ids if p not in exclude]
        return self.rng.choice(pool, size=min(n, len(pool)), replace=False).tolist()

    def _ts(self):
        offset = timedelta(seconds=self.batch_num * 30 + int(self.rng.integers(0, 30)))
        return (self.base_time + offset).strftime("%Y-%m-%dT%H:%M:%S.000Z")

    def _amt(self, low=PATTERN_AMT_MIN, high=PATTERN_AMT_MAX):
        return round(float(self.rng.uniform(low, high)), 2)

    def _next_id(self):
        self.tran_counter += 1
        return self.tran_counter

    def _make_normal_txns(self, n):
        src_idx = self.rng.zipf(a=1.5, size=n) % self.n_parties
        dst_idx = self.rng.zipf(a=1.3, size=n) % self.n_parties
        self_loops = src_idx == dst_idx
        dst_idx[self_loops] = (dst_idx[self_loops] + 1) % self.n_parties

        amounts = self.rng.lognormal(AMOUNT_MU, AMOUNT_SIGMA, size=n)
        amounts = np.clip(amounts, AMOUNT_MIN, AMOUNT_MAX).round(2)
        tx_types = self.rng.choice(TX_TYPES, size=n, p=TX_PROBS)

        txns = []
        for i in range(n):
            txns.append({
                "tran_id": self._next_id(),
                "tx_type": tx_types[i],
                "base_amt": float(amounts[i]),
                "tran_timestamp": self._ts(),
                "src": self.party_ids[src_idx[i]],
                "dst": self.party_ids[dst_idx[i]],
                "_is_pattern": False,
                "_alert_type": None,
                "_alert_id": None,
            })
        return txns

    def _inject_fan_out(self):
        n_legs = int(self.rng.integers(FAN_MIN_LEGS, FAN_MAX_LEGS + 1))
        parties = self._pick(1 + n_legs)
        sender, receivers = parties[0], parties[1:]
        aid = self.alert_counter; self.alert_counter += 1
        txns = []
        for recv in receivers:
            txns.append({
                "tran_id": self._next_id(), "tx_type": "TRANSFER-FanOut",
                "base_amt": self._amt(), "tran_timestamp": self._ts(),
                "src": sender, "dst": recv,
                "_is_pattern": True, "_alert_type": "fan_out", "_alert_id": aid,
            })
        return txns

    def _inject_fan_in(self):
        n_legs = int(self.rng.integers(FAN_MIN_LEGS, FAN_MAX_LEGS + 1))
        parties = self._pick(1 + n_legs)
        receiver, senders = parties[0], parties[1:]
        aid = self.alert_counter; self.alert_counter += 1
        txns = []
        for s in senders:
            txns.append({
                "tran_id": self._next_id(), "tx_type": "TRANSFER-FanIn",
                "base_amt": self._amt(), "tran_timestamp": self._ts(),
                "src": s, "dst": receiver,
                "_is_pattern": True, "_alert_type": "fan_in", "_alert_id": aid,
            })
        return txns

    def _inject_cycle(self):
        a, b, c = self._pick(3)
        aid = self.alert_counter; self.alert_counter += 1
        base = self._amt()
        txns = []
        for s, d in [(a, b), (b, c), (c, a)]:
            txns.append({
                "tran_id": self._next_id(), "tx_type": "TRANSFER-Forward",
                "base_amt": round(base * float(self.rng.uniform(0.90, 1.10)), 2),
                "tran_timestamp": self._ts(), "src": s, "dst": d,
                "_is_pattern": True, "_alert_type": "cycle", "_alert_id": aid,
            })
        return txns

    def _inject_scatter_gather(self):
        n_mids = 4
        parties = self._pick(2 + n_mids)
        source, collector = parties[0], parties[1]
        mids = parties[2:]
        aid = self.alert_counter; self.alert_counter += 1
        txns = []
        for mid in mids:
            amt = self._amt()
            txns.append({
                "tran_id": self._next_id(), "tx_type": "TRANSFER-FanOut",
                "base_amt": amt, "tran_timestamp": self._ts(),
                "src": source, "dst": mid,
                "_is_pattern": True, "_alert_type": "scatter_gather", "_alert_id": aid,
            })
            txns.append({
                "tran_id": self._next_id(), "tx_type": "TRANSFER-FanIn",
                "base_amt": round(amt * float(self.rng.uniform(0.90, 0.95)), 2),
                "tran_timestamp": self._ts(), "src": mid, "dst": collector,
                "_is_pattern": True, "_alert_type": "scatter_gather", "_alert_id": aid,
            })
        return txns

    def _inject_structuring(self):
        parties = self._pick(2)
        sender, recv = parties[0], parties[1]
        aid = self.alert_counter; self.alert_counter += 1
        txns = []
        for _ in range(STRUCTURING_NUM_TXNS):
            txns.append({
                "tran_id": self._next_id(), "tx_type": "TRANSFER-Periodical",
                "base_amt": self._amt(STRUCTURING_AMT_MIN, STRUCTURING_AMT_MAX),
                "tran_timestamp": self._ts(), "src": sender, "dst": recv,
                "_is_pattern": True, "_alert_type": "structuring", "_alert_id": aid,
            })
        return txns

    def next_batch(self, size=50):
        self.batch_num += 1
        txns = self._make_normal_txns(size)
        if self.rng.random() < self.pattern_prob:
            ptype = self.rng.choice(self.PATTERN_TYPES)
            injector = {
                "fan_out": self._inject_fan_out,
                "fan_in": self._inject_fan_in,
                "cycle": self._inject_cycle,
                "scatter_gather": self._inject_scatter_gather,
                "structuring": self._inject_structuring,
            }[ptype]
            txns.extend(injector())
            self.total_patterns_injected += 1
        return txns

print("TransactionStream defined (5 pattern types)")

TransactionStream defined (5 pattern types)


In [ ]:
# ═══════════════════════════════════════════════════════════
#  LiveGraph (from NB 11)
# ═══════════════════════════════════════════════════════════

class LiveGraph:
    """Maintains graph state with O(k) incremental updates per batch."""

    FEATURE_COLS = [
        "type", "in_degree", "out_degree",
        "total_amount_sent", "total_amount_received",
        "avg_amount_sent", "avg_amount_received",
        "unique_counterparties_sent", "unique_counterparties_received",
    ]

    def __init__(self):
        nf = pd.read_parquet(os.path.join(DATA_DIR, "node_features.parquet"))
        edges_df = pd.read_parquet(os.path.join(DATA_DIR, "edges.parquet"))

        self.node_ids = list(nf["id"].values)
        self.node_to_idx = {nid: i for i, nid in enumerate(self.node_ids)}
        self.n_nodes = len(self.node_ids)

        self.node_type = {}
        self.out_degree = defaultdict(int)
        self.in_degree = defaultdict(int)
        self.total_sent = defaultdict(float)
        self.total_received = defaultdict(float)
        self.counterparties_sent = defaultdict(set)
        self.counterparties_received = defaultdict(set)

        for _, row in nf.iterrows():
            nid = row["id"]
            self.node_type[nid] = int(row["type"])
            self.out_degree[nid] = int(row["out_degree"])
            self.in_degree[nid] = int(row["in_degree"])
            self.total_sent[nid] = float(row["total_amount_sent"])
            self.total_received[nid] = float(row["total_amount_received"])
            self.counterparties_sent[nid] = set(range(int(row["unique_counterparties_sent"])))
            self.counterparties_received[nid] = set(range(int(row["unique_counterparties_received"])))

        self.edges = set()
        for _, row in edges_df.iterrows():
            src_idx = self.node_to_idx.get(row["source"])
            dst_idx = self.node_to_idx.get(row["target"])
            if src_idx is not None and dst_idx is not None:
                self.edges.add((src_idx, dst_idx))

        self.is_sar = {row["id"]: int(row.get("is_sar", 0)) for _, row in nf.iterrows()}
        self.total_txns_processed = 0
        self.pattern_nodes = set()

    def update(self, txn_batch):
        affected = set()
        for txn in txn_batch:
            src = txn["src"]
            dst = txn["dst"]
            amt = txn["base_amt"]

            for nid in (src, dst):
                if nid not in self.node_to_idx:
                    idx = self.n_nodes
                    self.node_to_idx[nid] = idx
                    self.node_ids.append(nid)
                    self.n_nodes += 1
                    self.node_type[nid] = 0

            src_idx = self.node_to_idx[src]
            dst_idx = self.node_to_idx[dst]

            self.out_degree[src] += 1
            self.in_degree[dst] += 1
            self.total_sent[src] += amt
            self.total_received[dst] += amt
            self.counterparties_sent[src].add(dst_idx)
            self.counterparties_received[dst].add(src_idx)
            self.edges.add((src_idx, dst_idx))
            affected.add(src_idx)
            affected.add(dst_idx)

            if txn.get("_is_pattern"):
                self.pattern_nodes.add(src)
                self.pattern_nodes.add(dst)
                self.is_sar[src] = 1
                self.is_sar[dst] = 1

        self.total_txns_processed += len(txn_batch)
        return affected

    def get_feature_matrix(self):
        features = np.zeros((self.n_nodes, 9), dtype=np.float32)
        for i, nid in enumerate(self.node_ids):
            od = self.out_degree[nid]
            ind = self.in_degree[nid]
            ts = self.total_sent[nid]
            tr = self.total_received[nid]
            features[i, 0] = self.node_type.get(nid, 0)
            features[i, 1] = ind
            features[i, 2] = od
            features[i, 3] = ts
            features[i, 4] = tr
            features[i, 5] = (ts / od) if od > 0 else 0.0
            features[i, 6] = (tr / ind) if ind > 0 else 0.0
            features[i, 7] = len(self.counterparties_sent[nid])
            features[i, 8] = len(self.counterparties_received[nid])
        return torch.tensor(features, dtype=torch.float32)

    def get_edge_index(self):
        if not self.edges:
            return torch.zeros((2, 0), dtype=torch.long)
        src_list, dst_list = zip(*self.edges)
        return torch.tensor([list(src_list), list(dst_list)], dtype=torch.long)

    def get_sar_labels(self):
        return np.array([self.is_sar.get(nid, 0) for nid in self.node_ids])

print("LiveGraph class defined (incremental O(k) updates)")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  RealtimeScorer (from NB 11)
# ═══════════════════════════════════════════════════════════

class RealtimeScorer:
    """Scores nodes using pre-trained GraphSAGE + WGAN-GP."""

    def __init__(self, device, threshold_pct=99, rescore_interval=3):
        self.device = device
        self.rescore_interval = rescore_interval

        with open(os.path.join(MODELS_DIR, "training_meta.json")) as f:
            self.meta = json.load(f)

        self.sage = GraphSAGE(9, 128, 64, dropout=0.3).to(device)
        self.sage.load_state_dict(torch.load(os.path.join(MODELS_DIR, "graphsage.pt"), map_location=device, weights_only=True))
        self.sage.eval()

        m = self.meta
        self.encoder = Encoder(m["input_dim"], m["latent_dim"], m["e_hidden"], m["n_layers"], m["activation"]).to(device)
        self.generator = Generator(m["latent_dim"], m["input_dim"], m["g_hidden"], m["n_layers"], m["activation"]).to(device)
        self.encoder.load_state_dict(torch.load(os.path.join(MODELS_DIR, "encoder.pt"), map_location=device, weights_only=True))
        self.generator.load_state_dict(torch.load(os.path.join(MODELS_DIR, "generator.pt"), map_location=device, weights_only=True))
        self.encoder.eval()
        self.generator.eval()

        norm = torch.load(os.path.join(MODELS_DIR, "feature_norm.pt"), map_location=device, weights_only=True)
        self.x_mean = norm["mean"].to(device)
        self.x_std = norm["std"].to(device)

        X_train = np.load(os.path.join(MODELS_DIR, "X_train.npy"))
        train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
        train_scores = anomaly_score(train_tensor, self.encoder, self.generator).cpu().numpy()
        self.threshold = float(np.percentile(train_scores, threshold_pct))
        del train_tensor, train_scores
        torch.cuda.empty_cache()

        self.all_scores = None
        self.all_embeddings = None
        self.alerts = deque(maxlen=200)
        self.score_history = deque(maxlen=200)
        self.latency_history = []
        self.total_alerts = 0
        self.total_scored = 0

    def score(self, graph, affected_indices, batch_num, txn_batch):
        t0 = time.perf_counter()
        should_rescore = (batch_num % self.rescore_interval == 0) or (self.all_scores is None)

        if should_rescore:
            x = graph.get_feature_matrix().to(self.device)
            edge_index = graph.get_edge_index().to(self.device)
            x_norm = (x - self.x_mean) / self.x_std
            embeddings = self.sage.full_forward(x_norm, edge_index)
            self.all_embeddings = embeddings.cpu()
            scores = anomaly_score(embeddings, self.encoder, self.generator)
            self.all_scores = scores.cpu().numpy()
            del x, edge_index, x_norm, embeddings, scores
            torch.cuda.empty_cache()

        latency_ms = (time.perf_counter() - t0) * 1000
        self.latency_history.append(latency_ms)

        affected_list = sorted(affected_indices)
        batch_alerts = []

        if self.all_scores is not None:
            for idx in affected_list:
                if idx < len(self.all_scores):
                    score_val = float(self.all_scores[idx])
                    node_id = graph.node_ids[idx]
                    if score_val > self.threshold:
                        alert = {
                            "batch": batch_num,
                            "node_id": node_id,
                            "score": score_val,
                            "is_sar": graph.is_sar.get(node_id, 0),
                            "timestamp": datetime.now().strftime("%H:%M:%S"),
                        }
                        self.alerts.append(alert)
                        batch_alerts.append(alert)
                        self.total_alerts += 1

            self.total_scored += len(affected_list)
            affected_scores = [self.all_scores[i] for i in affected_list if i < len(self.all_scores)]
            if affected_scores:
                self.score_history.append((
                    batch_num,
                    float(np.mean(affected_scores)),
                    float(np.max(affected_scores)),
                    len(batch_alerts),
                ))

        return {
            "batch_num": batch_num,
            "affected_count": len(affected_list),
            "alerts": batch_alerts,
            "latency_ms": latency_ms,
            "rescored": should_rescore,
        }

print("RealtimeScorer class defined")

In [7]:
# ═══════════════════════════════════════════════════════════
#  SimulationState — thread-safe shared state (NEW)
# ═══════════════════════════════════════════════════════════

class SimulationState:
    """Thread-safe shared state between simulation thread and Dash callbacks."""

    def __init__(self):
        self._lock = threading.Lock()
        self._state = self._initial_state()

    def _initial_state(self):
        return {
            "status": "idle",
            "batch_num": 0,
            "total_batches": 0,
            "total_txns": 0,
            "total_alerts": 0,
            "total_patterns": 0,
            "pattern_nodes": 0,
            "score_history": [],
            "recent_alerts": [],
            "all_scores": None,
            "sar_labels": None,
            "threshold": 0.0,
            "latency_history": [],
            "confusion": {"tp": 0, "fp": 0, "fn": 0, "tn": 0},
            "elapsed": 0.0,
            "last_batch_info": "",
            "rescore_interval": 3,
            "top_alert": None,
            "ego_edges": [],
            "ego_node_meta": {},
            "ego_node_count": 0,
            "error": None,
        }

    def get(self):
        """Return a snapshot of the current state (safe for Dash thread)."""
        with self._lock:
            snap = {}
            for k, v in self._state.items():
                if isinstance(v, np.ndarray):
                    snap[k] = v.copy()
                elif isinstance(v, (list, dict)):
                    snap[k] = deepcopy(v)
                else:
                    snap[k] = v
            return snap

    def update(self, **kwargs):
        """Update one or more keys atomically."""
        with self._lock:
            self._state.update(kwargs)

    def reset(self):
        """Reset to initial state."""
        with self._lock:
            self._state = self._initial_state()

sim_state = SimulationState()
print("SimulationState ready (thread-safe shared state)")

SimulationState ready (thread-safe shared state)


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Simulation Thread + GPU Cleanup (NEW)
# ═══════════════════════════════════════════════════════════

_stream = None
_graph = None
_scorer = None
_stop_event = threading.Event()
_sim_thread = None


def _cleanup_gpu():
    """Free GPU memory from scorer models."""
    global _scorer, _graph, _stream
    if _scorer is not None:
        try:
            del _scorer.sage, _scorer.encoder, _scorer.generator
            del _scorer
        except Exception:
            pass
        _scorer = None
    _graph = None
    _stream = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


MAX_EGO_NEIGHBORS = 30  # Cap to keep Plotly rendering fast


def _build_ego_graph_state():
    """After sim completes, compute ego-graph for the top alert node (capped)."""
    if not _scorer or not _scorer.alerts:
        sim_state.update(top_alert=None)
        return
    top_alert = max(_scorer.alerts, key=lambda a: a["score"])
    top_node = top_alert["node_id"]
    top_idx = _graph.node_to_idx.get(top_node)
    if top_idx is None:
        sim_state.update(top_alert=dict(top_alert), ego_edges=[], ego_node_meta={}, ego_node_count=0)
        return

    # Build adjacency only for the center node's direct neighbors
    adj_out = defaultdict(set)
    adj_in = defaultdict(set)
    for s, d in _graph.edges:
        if s == top_idx or d == top_idx:
            adj_out[s].add(d)
            adj_in[d].add(s)

    all_neighbors = list(adj_out.get(top_idx, set()) | adj_in.get(top_idx, set()))

    # Prioritize: SAR neighbors first, then by anomaly score if available
    def _neighbor_priority(idx):
        nid = _graph.node_ids[idx] if idx < len(_graph.node_ids) else ""
        is_sar = _graph.is_sar.get(nid, 0)
        score = float(_scorer.all_scores[idx]) if (_scorer.all_scores is not None and idx < len(_scorer.all_scores)) else 0.0
        return (-is_sar, -score)  # SAR first, then highest score

    all_neighbors.sort(key=_neighbor_priority)
    selected_neighbors = all_neighbors[:MAX_EGO_NEIGHBORS]
    subgraph_indices = {top_idx} | set(selected_neighbors)
    idx_to_node = {i: _graph.node_ids[i] for i in subgraph_indices if i < len(_graph.node_ids)}

    # Rebuild full adjacency for selected subgraph only
    sub_adj_out = defaultdict(set)
    for s, d in _graph.edges:
        if s in subgraph_indices and d in subgraph_indices:
            sub_adj_out[s].add(d)

    ego_edges = []
    for s_idx in subgraph_indices:
        for d_idx in sub_adj_out.get(s_idx, set()):
            ego_edges.append((idx_to_node.get(s_idx, str(s_idx))[:6],
                              idx_to_node.get(d_idx, str(d_idx))[:6]))

    ego_node_meta = {}
    for idx in subgraph_indices:
        nid = idx_to_node.get(idx, str(idx))
        short = nid[:6]
        score = float(_scorer.all_scores[idx]) if (_scorer.all_scores is not None and idx < len(_scorer.all_scores)) else 0.0
        ego_node_meta[short] = {
            "is_center": (idx == top_idx),
            "is_sar": bool(_graph.is_sar.get(nid, 0)),
            "full_id": nid,
            "score": round(score, 4),
        }

    sim_state.update(
        top_alert=dict(top_alert),
        ego_edges=ego_edges,
        ego_node_meta=ego_node_meta,
        ego_node_count=len(subgraph_indices),
    )


def simulation_loop(config):
    """Runs in a background thread. Reads config dict, writes to sim_state."""
    global _stream, _graph, _scorer

    try:
        # Party CSV: try DATA_DIR first, fall back to ../demodata/
        _party_path = os.path.join(DATA_DIR, "party.csv")
        if not os.path.exists(_party_path):
            _party_path = os.path.join("..", "demodata", "party.csv")
        party_df = pd.read_csv(_party_path)
        party_ids = party_df["partyId"].tolist()
        rng = np.random.default_rng(config["SEED"])

        _stream = TransactionStream(party_ids, rng, pattern_prob=config["PATTERN_PROB"])
        _graph = LiveGraph()
        _scorer = RealtimeScorer(device, threshold_pct=config["THRESHOLD_PERCENTILE"],
                                 rescore_interval=config["RESCORE_INTERVAL"])

        sim_state.update(
            status="running",
            threshold=_scorer.threshold,
            total_batches=config["NUM_BATCHES"],
            rescore_interval=config["RESCORE_INTERVAL"],
        )

        sim_start = time.perf_counter()

        for batch_num in range(1, config["NUM_BATCHES"] + 1):
            if _stop_event.is_set():
                _build_ego_graph_state()
                sim_state.update(status="stopped", elapsed=time.perf_counter() - sim_start)
                return

            txn_batch = _stream.next_batch(config["BATCH_SIZE"])
            affected = _graph.update(txn_batch)
            result = _scorer.score(_graph, affected, batch_num, txn_batch)

            # Compute confusion matrix
            confusion = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
            if _scorer.all_scores is not None:
                sar_labels = _graph.get_sar_labels()
                n_scores = min(len(_scorer.all_scores), len(sar_labels))
                predicted = (_scorer.all_scores[:n_scores] > _scorer.threshold).astype(int)
                actual = sar_labels[:n_scores]
                confusion["tp"] = int(((predicted == 1) & (actual == 1)).sum())
                confusion["fp"] = int(((predicted == 1) & (actual == 0)).sum())
                confusion["fn"] = int(((predicted == 0) & (actual == 1)).sum())
                confusion["tn"] = int(((predicted == 0) & (actual == 0)).sum())

            recent = [dict(a) for a in list(_scorer.alerts)[-20:]]
            n_pattern = sum(1 for t in txn_batch if t.get("_is_pattern"))

            sim_state.update(
                batch_num=batch_num,
                total_txns=_graph.total_txns_processed,
                total_alerts=_scorer.total_alerts,
                total_patterns=_stream.total_patterns_injected,
                pattern_nodes=len(_graph.pattern_nodes),
                score_history=list(_scorer.score_history),
                recent_alerts=recent,
                all_scores=_scorer.all_scores.copy() if _scorer.all_scores is not None else None,
                sar_labels=_graph.get_sar_labels(),
                latency_history=list(_scorer.latency_history),
                confusion=confusion,
                elapsed=time.perf_counter() - sim_start,
                last_batch_info=(
                    f"Batch {batch_num}/{config['NUM_BATCHES']}: {len(txn_batch)} txns "
                    f"({n_pattern} pattern) | "
                    f"{result['affected_count']} affected | "
                    f"{len(result['alerts'])} new alerts | "
                    f"{result['latency_ms']:.0f}ms"
                    f"{' [RESCORE]' if result['rescored'] else ''}"
                ),
            )

            # Interruptible sleep
            if batch_num < config["NUM_BATCHES"]:
                if _stop_event.wait(timeout=config["BATCH_INTERVAL"]):
                    _build_ego_graph_state()
                    sim_state.update(status="stopped", elapsed=time.perf_counter() - sim_start)
                    return

        _build_ego_graph_state()
        sim_state.update(status="finished", elapsed=time.perf_counter() - sim_start)

    except Exception as e:
        import traceback
        traceback.print_exc()
        sim_state.update(status="stopped", error=str(e))


print("Simulation thread functions defined")

In [9]:
# ═══════════════════════════════════════════════════════════
#  Dash App Layout (NEW)
# ═══════════════════════════════════════════════════════════

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

CLR = dict(blue="#3498db", green="#2ecc71", red="#e74c3c", purple="#9b59b6",
           orange="#e67e22", yellow="#f1c40f", dark="#2c3e50", light="#ecf0f1")


def kpi_card(title, value, color, card_id):
    return dbc.Card([
        dbc.CardBody([
            html.H6(title, className="text-muted mb-1", style={"fontSize": "0.85rem"}),
            html.H3(value, id=card_id, className="mb-0",
                     style={"color": color, "fontWeight": "bold"}),
        ])
    ], className="shadow-sm", style={"borderLeft": f"4px solid {color}"})


# ---- Tab 1: Control & Monitor ----
config_panel = dbc.Card([
    dbc.CardHeader("Simulation Configuration", style={"fontWeight": "bold"}),
    dbc.CardBody([
        dbc.Row([
            dbc.Col([
                dbc.Label("Batches", size="sm"),
                dbc.Input(id="cfg-batches", type="number", value=80, min=1, max=500, size="sm"),
            ], md=2),
            dbc.Col([
                dbc.Label("Batch Size", size="sm"),
                dbc.Input(id="cfg-batch-size", type="number", value=50, min=10, max=500, size="sm"),
            ], md=2),
            dbc.Col([
                dbc.Label("Pattern %", size="sm"),
                dbc.Input(id="cfg-pattern-pct", type="number", value=10, min=0, max=100, size="sm"),
            ], md=2),
            dbc.Col([
                dbc.Label("Rescore Every", size="sm"),
                dbc.Input(id="cfg-rescore", type="number", value=3, min=1, max=20, size="sm"),
            ], md=2),
            dbc.Col([
                dbc.Label("Threshold P", size="sm"),
                dbc.Input(id="cfg-threshold", type="number", value=99, min=80, max=100, size="sm"),
            ], md=2),
            dbc.Col([
                dbc.Label("Interval (s)", size="sm"),
                dbc.Input(id="cfg-interval", type="number", value=0.5, min=0.1, max=5.0, step=0.1, size="sm"),
            ], md=2),
        ], className="g-2"),
        html.Hr(style={"margin": "12px 0"}),
        dbc.Row([
            dbc.Col([
                dbc.Button("START", id="btn-start", color="success", className="me-2", size="lg"),
                dbc.Button("STOP", id="btn-stop", color="danger", className="me-2", size="lg", disabled=True),
                dbc.Button("RESET", id="btn-reset", color="secondary", size="lg"),
            ], className="text-center"),
        ]),
    ]),
], className="mb-3")

kpi_row = dbc.Row([
    dbc.Col(kpi_card("Batches", "0/0", CLR["blue"], "kpi-batches"), md=3),
    dbc.Col(kpi_card("Transactions", "0", CLR["green"], "kpi-txns"), md=3),
    dbc.Col(kpi_card("Alerts", "0", CLR["red"], "kpi-alerts"), md=3),
    dbc.Col(kpi_card("Patterns", "0", CLR["purple"], "kpi-patterns"), md=3),
], className="mb-3 g-3")

charts_row_1 = dbc.Row([
    dbc.Col(dcc.Graph(id="chart-scores", config={"displayModeBar": False},
                      style={"height": "350px"}), md=7),
    dbc.Col([
        html.H6("Alert Feed", className="text-center mb-2"),
        dash_table.DataTable(
            id="table-alerts",
            columns=[
                {"name": "Time", "id": "timestamp"},
                {"name": "Node", "id": "node_id"},
                {"name": "Score", "id": "score", "type": "numeric",
                 "format": Format(precision=2, scheme=Scheme.fixed)},
                {"name": "SAR", "id": "is_sar"},
            ],
            data=[],
            page_size=10,
            style_header={"backgroundColor": "#2c3e50", "color": "white",
                          "fontWeight": "bold", "fontSize": "12px"},
            style_cell={"backgroundColor": "#1a1a2e", "color": "white",
                        "fontSize": "11px", "padding": "4px 8px"},
            style_data_conditional=[
                {"if": {"filter_query": "{is_sar} = 1"},
                 "backgroundColor": "rgba(220,38,38,0.2)"},
            ],
        ),
    ], md=5),
], className="mb-3")

charts_row_2 = dbc.Row([
    dbc.Col(dcc.Graph(id="chart-confusion", config={"displayModeBar": False},
                      style={"height": "320px"}), md=5),
    dbc.Col(dcc.Graph(id="chart-histogram", config={"displayModeBar": False},
                      style={"height": "320px"}), md=7),
], className="mb-3")

tab1 = dbc.Container([
    config_panel,
    kpi_row,
    html.P(id="batch-info", className="text-muted text-center mb-2",
           style={"fontSize": "0.9rem", "fontFamily": "monospace"}),
    dbc.Alert(id="error-alert", color="danger", is_open=False, dismissable=True),
    charts_row_1,
    charts_row_2,
], fluid=True)

# ---- Tab 2: Analysis ----
tab2 = dbc.Container([
    dbc.Alert("Run a simulation first, then review analysis here.",
              id="analysis-alert", color="info", is_open=True),
    dbc.Row([
        dbc.Col(kpi_card("Duration", "--", CLR["blue"], "kpi-duration"), md=2),
        dbc.Col(kpi_card("Throughput", "--", CLR["green"], "kpi-throughput"), md=2),
        dbc.Col(kpi_card("Precision", "--", CLR["orange"], "kpi-precision"), md=2),
        dbc.Col(kpi_card("Recall", "--", CLR["green"], "kpi-recall"), md=2),
        dbc.Col(kpi_card("F1 Score", "--", CLR["purple"], "kpi-f1"), md=2),
        dbc.Col(kpi_card("P95 Latency", "--", CLR["red"], "kpi-p95"), md=2),
    ], className="mb-3 g-3"),
    dbc.Row([
        dbc.Col(dcc.Graph(id="chart-latency-bar", style={"height": "380px"}), md=6),
        dbc.Col(dcc.Graph(id="chart-latency-hist", style={"height": "380px"}), md=6),
    ]),
], fluid=True)

# ---- Tab 3: Investigation ----
tab3 = dbc.Container([
    dbc.Alert("Investigation available after simulation completes or is stopped.",
              id="investigation-alert", color="info", is_open=True),
    dbc.Row([
        dbc.Col([
            html.H5("Top Alert Node", className="mb-3"),
            dbc.Button("Refresh", id="btn-refresh-investigation", color="primary",
                       size="sm", className="mb-3"),
            html.Div(id="top-alert-info"),
        ], md=3),
        dbc.Col([
            dcc.Graph(id="chart-ego-graph", style={"height": "520px"}),
        ], md=9),
    ]),
], fluid=True)

# ---- App Layout ----
app.layout = dbc.Container([
    dcc.Interval(id="interval-component", interval=500, n_intervals=0, disabled=True),
    dcc.Store(id="store-status", data="idle"),
    dbc.Row([dbc.Col([
        html.H2([
            "AML Real-Time Simulation Dashboard ",
            html.Span(id="status-badge"),
        ], className="text-center mt-3 mb-1"),
        html.Div([
            dbc.Badge("GraphSAGE + WGAN-GP", color="info", className="me-2"),
            dbc.Badge("PyTorch + Dash", color="primary", className="me-2"),
        ], className="text-center mb-3"),
    ], md=12)]),
    dbc.Tabs([
        dbc.Tab(tab1, label="Control & Monitor", tab_id="tab-control"),
        dbc.Tab(tab2, label="Analysis", tab_id="tab-analysis"),
        dbc.Tab(tab3, label="Investigation", tab_id="tab-investigation"),
    ], id="tabs", active_tab="tab-control"),
], fluid=True)

print("Dash layout defined (3 tabs)")


Dash layout defined (3 tabs)


In [10]:
# ═══════════════════════════════════════════════════════════
#  Callback: Start / Stop / Reset (NEW)
# ═══════════════════════════════════════════════════════════

@app.callback(
    [Output("interval-component", "disabled"),
     Output("store-status", "data"),
     Output("btn-start", "disabled"),
     Output("btn-stop", "disabled"),
     Output("btn-reset", "disabled"),
     Output("cfg-batches", "disabled"),
     Output("cfg-batch-size", "disabled"),
     Output("cfg-pattern-pct", "disabled"),
     Output("cfg-rescore", "disabled"),
     Output("cfg-threshold", "disabled"),
     Output("cfg-interval", "disabled")],
    [Input("btn-start", "n_clicks"),
     Input("btn-stop", "n_clicks"),
     Input("btn-reset", "n_clicks")],
    [State("cfg-batches", "value"),
     State("cfg-batch-size", "value"),
     State("cfg-pattern-pct", "value"),
     State("cfg-rescore", "value"),
     State("cfg-threshold", "value"),
     State("cfg-interval", "value"),
     State("store-status", "data")],
    prevent_initial_call=True,
)
def handle_buttons(start_clicks, stop_clicks, reset_clicks,
                   batches, batch_size, pattern_pct, rescore, threshold_p, interval,
                   current_status):
    global _sim_thread, _stop_event

    triggered = ctx.triggered_id
    # Outputs: interval_disabled, store_status, start_dis, stop_dis, reset_dis, 6x cfg_dis

    if triggered == "btn-start":
        if current_status in ("running", "stopping"):
            raise dash.exceptions.PreventUpdate

        # Clean up any previous run
        _stop_event.set()
        if _sim_thread and _sim_thread.is_alive():
            _sim_thread.join(timeout=2.0)
        _cleanup_gpu()

        _stop_event = threading.Event()
        config = {
            "NUM_BATCHES": int(batches or 80),
            "BATCH_SIZE": int(batch_size or 50),
            "BATCH_INTERVAL": float(interval or 0.5),
            "PATTERN_PROB": float(pattern_pct or 10) / 100.0,
            "RESCORE_INTERVAL": int(rescore or 3),
            "THRESHOLD_PERCENTILE": int(threshold_p or 99),
            "SEED": 42,
        }
        sim_state.reset()
        sim_state.update(status="running", total_batches=config["NUM_BATCHES"])

        _sim_thread = threading.Thread(target=simulation_loop, args=(config,), daemon=True)
        _sim_thread.start()

        # interval ON, start OFF, stop ON, reset OFF, configs OFF
        return (False, "running", True, False, True, True, True, True, True, True, True)

    elif triggered == "btn-stop":
        _stop_event.set()
        sim_state.update(status="stopping")
        # Keep interval ON so update_live/investigation catch final ego-graph state
        # interval ON, start OFF, stop OFF, reset ON, configs ON
        return (False, "stopping", True, True, False, False, False, False, False, False, False)

    elif triggered == "btn-reset":
        _stop_event.set()
        if _sim_thread and _sim_thread.is_alive():
            _sim_thread.join(timeout=2.0)
        _cleanup_gpu()
        sim_state.reset()
        # interval OFF, start ON, stop OFF, reset ON, configs ON
        return (True, "idle", False, True, False, False, False, False, False, False, False)

    raise dash.exceptions.PreventUpdate


print("Button callbacks registered")


Button callbacks registered


In [11]:
# ═══════════════════════════════════════════════════════════
#  Callback: Live Dashboard Update (NEW)
# ═══════════════════════════════════════════════════════════

def _status_badge(status):
    color_map = {"idle": "secondary", "running": "success",
                 "stopped": "warning", "finished": "info"}
    return dbc.Badge(status.upper(), color=color_map.get(status, "secondary"),
                     className="ms-2", style={"fontSize": "1rem"})


@app.callback(
    [Output("kpi-batches", "children"),
     Output("kpi-txns", "children"),
     Output("kpi-alerts", "children"),
     Output("kpi-patterns", "children"),
     Output("batch-info", "children"),
     Output("status-badge", "children"),
     Output("error-alert", "children"),
     Output("error-alert", "is_open"),
     Output("chart-scores", "figure"),
     Output("table-alerts", "data"),
     Output("chart-confusion", "figure"),
     Output("chart-histogram", "figure"),
     # Auto-stop interval when sim ends
     Output("interval-component", "disabled", allow_duplicate=True),
     Output("btn-start", "disabled", allow_duplicate=True),
     Output("btn-stop", "disabled", allow_duplicate=True),
     Output("btn-reset", "disabled", allow_duplicate=True)],
    [Input("interval-component", "n_intervals")],
    prevent_initial_call=True,
)
def update_live(n_intervals):
    state = sim_state.get()

    # KPIs
    kpi_batches = f"{state['batch_num']}/{state['total_batches']}"
    kpi_txns = f"{state['total_txns']:,}"
    kpi_alerts = f"{state['total_alerts']:,}"
    kpi_patterns = str(state['total_patterns'])
    batch_info = state["last_batch_info"]
    badge = _status_badge(state["status"])

    # Error alert
    error_msg = state.get("error") or ""
    error_open = bool(error_msg)

    # ---- Score Time Series ----
    fig_scores = go.Figure()
    if state["score_history"]:
        batches_arr = [s[0] for s in state["score_history"]]
        means = [s[1] for s in state["score_history"]]
        maxes = [s[2] for s in state["score_history"]]

        fig_scores.add_trace(go.Scatter(
            x=batches_arr, y=means, mode="lines", name="Mean Score",
            fill="tozeroy", fillcolor="rgba(52,152,219,0.15)",
            line=dict(color=CLR["blue"], width=2),
        ))
        fig_scores.add_trace(go.Scatter(
            x=batches_arr, y=maxes, mode="lines", name="Max Score",
            line=dict(color=CLR["red"], width=1, dash="dot"),
        ))
        if state["threshold"] > 0:
            fig_scores.add_hline(
                y=state["threshold"], line_dash="dash", line_color=CLR["orange"],
                annotation_text=f"Threshold={state['threshold']:.2f}",
            )
        # Alert markers
        alert_b = [s[0] for s in state["score_history"] if s[3] > 0]
        alert_m = [s[2] for s in state["score_history"] if s[3] > 0]
        if alert_b:
            fig_scores.add_trace(go.Scatter(
                x=alert_b, y=alert_m, mode="markers", name="Alert Batch",
                marker=dict(color=CLR["red"], size=8, symbol="triangle-down"),
            ))
    fig_scores.update_layout(
        template="plotly_dark", title="Score Time Series",
        xaxis_title="Batch", yaxis_title="Anomaly Score",
        margin=dict(t=40, b=30, l=50, r=10),
        legend=dict(orientation="h", y=-0.18),
        height=350,
    )

    # ---- Alert Table ----
    alert_data = []
    for a in reversed(state["recent_alerts"]):
        alert_data.append({
            "timestamp": a.get("timestamp", ""),
            "node_id": a.get("node_id", "")[:10],
            "score": round(a.get("score", 0), 2),
            "is_sar": a.get("is_sar", 0),
        })

    # ---- Confusion Matrix ----
    cm = state["confusion"]
    cm_array = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
    precision = cm["tp"] / max(cm["tp"] + cm["fp"], 1)
    recall = cm["tp"] / max(cm["tp"] + cm["fn"], 1)

    fig_cm = go.Figure(data=go.Heatmap(
        z=cm_array, x=["Pred: Normal", "Pred: Anomaly"],
        y=["Actual: Normal", "Actual: SAR"],
        text=[[f"{cm_array[i][j]:,}" for j in range(2)] for i in range(2)],
        texttemplate="%{text}", textfont={"size": 16},
        colorscale="YlOrRd", showscale=False,
    ))
    fig_cm.update_layout(
        template="plotly_dark",
        title=f"Confusion Matrix (P={precision:.2f} R={recall:.2f})",
        margin=dict(t=40, b=30, l=100, r=10),
        height=320,
    )

    # ---- Score Histogram ----
    fig_hist = go.Figure()
    if state["all_scores"] is not None and state["sar_labels"] is not None:
        scores = state["all_scores"]
        sars = state["sar_labels"]
        n = min(len(scores), len(sars))
        clip_val = np.percentile(scores[:n], 99.5) if n > 0 else 1.0
        scores_c = np.clip(scores[:n], 0, clip_val)
        sar_mask = sars[:n] == 1

        fig_hist.add_trace(go.Histogram(
            x=scores_c[~sar_mask], nbinsx=60,
            name=f"Normal ({(~sar_mask).sum():,})",
            marker_color=CLR["blue"], opacity=0.6,
        ))
        if sar_mask.sum() > 0:
            fig_hist.add_trace(go.Histogram(
                x=scores_c[sar_mask], nbinsx=40,
                name=f"SAR ({sar_mask.sum():,})",
                marker_color=CLR["red"], opacity=0.6,
            ))
        if state["threshold"] > 0:
            fig_hist.add_vline(x=state["threshold"], line_dash="dash",
                               line_color=CLR["orange"])
    fig_hist.update_layout(
        template="plotly_dark", title="Score Distribution",
        xaxis_title="Anomaly Score", yaxis_title="Count",
        barmode="overlay",
        margin=dict(t=40, b=30, l=50, r=10),
        height=320,
    )

    # Auto-disable interval when simulation fully ends
    status = state["status"]
    interval_disabled = status in ("idle", "stopped", "finished")
    start_disabled = status in ("running", "stopping")
    stop_disabled = status != "running"
    reset_disabled = status == "running"

    return (
        kpi_batches, kpi_txns, kpi_alerts, kpi_patterns,
        batch_info, badge, error_msg, error_open,
        fig_scores, alert_data, fig_cm, fig_hist,
        interval_disabled, start_disabled, stop_disabled, reset_disabled,
    )


print("Live update callback registered")


Live update callback registered


In [12]:
# ═══════════════════════════════════════════════════════════
#  Callbacks: Analysis + Investigation tabs (NEW)
# ═══════════════════════════════════════════════════════════

@app.callback(
    [Output("kpi-duration", "children"),
     Output("kpi-throughput", "children"),
     Output("kpi-precision", "children"),
     Output("kpi-recall", "children"),
     Output("kpi-f1", "children"),
     Output("kpi-p95", "children"),
     Output("chart-latency-bar", "figure"),
     Output("chart-latency-hist", "figure"),
     Output("analysis-alert", "is_open")],
    [Input("interval-component", "n_intervals"),
     Input("tabs", "active_tab")],
    prevent_initial_call=True,
)
def update_analysis(n_intervals, active_tab):
    state = sim_state.get()
    latencies = state["latency_history"]
    empty = go.Figure()
    empty.update_layout(template="plotly_dark", height=380)

    if not latencies or state["status"] == "idle":
        return ("--", "--", "--", "--", "--", "--", empty, empty, True)

    cm = state["confusion"]
    tp, fp, fn = cm["tp"], cm["fp"], cm["fn"]
    p = tp / max(tp + fp, 1)
    r = tp / max(tp + fn, 1)
    f1 = 2 * (p * r) / max(p + r, 1e-9)
    elapsed = max(state["elapsed"], 0.01)
    throughput = state["total_txns"] / elapsed
    p95 = float(np.percentile(latencies, 95))
    rescore_int = state.get("rescore_interval", 3)

    # Latency bar chart
    batch_nums = list(range(1, len(latencies) + 1))
    rescore_mask = [(i % rescore_int == 0 or i == 1) for i in batch_nums]
    colors_lat = [CLR["red"] if rm else CLR["blue"] for rm in rescore_mask]

    fig_bar = go.Figure(go.Bar(
        x=batch_nums, y=latencies, marker_color=colors_lat, opacity=0.7,
    ))
    fig_bar.add_hline(y=np.mean(latencies), line_dash="dash", line_color=CLR["green"],
                      annotation_text=f"Mean: {np.mean(latencies):.0f}ms")
    fig_bar.update_layout(
        template="plotly_dark",
        title="Scoring Latency per Batch (red=rescore, blue=cached)",
        xaxis_title="Batch", yaxis_title="Latency (ms)",
        margin=dict(t=40, b=30), height=380,
    )

    # Latency histogram
    fig_hist = go.Figure(go.Histogram(
        x=latencies, nbinsx=30, marker_color=CLR["blue"], opacity=0.7,
    ))
    fig_hist.add_vline(x=p95, line_dash="dash", line_color=CLR["red"],
                       annotation_text=f"P95: {p95:.0f}ms")
    fig_hist.add_vline(x=np.mean(latencies), line_dash="dash", line_color=CLR["green"],
                       annotation_text=f"Mean: {np.mean(latencies):.0f}ms")
    fig_hist.update_layout(
        template="plotly_dark", title="Latency Distribution",
        xaxis_title="Latency (ms)", yaxis_title="Count",
        margin=dict(t=40, b=30), height=380,
    )

    return (
        f"{elapsed:.1f}s", f"{throughput:.0f} txns/s",
        f"{p:.4f}", f"{r:.4f}", f"{f1:.4f}", f"{p95:.0f}ms",
        fig_bar, fig_hist, False,
    )


def _build_ego_plotly(ego_edges, ego_node_meta):
    """Convert ego-graph data to a Plotly figure."""
    G = nx.DiGraph()
    for src, dst in ego_edges:
        G.add_edge(src, dst)
    for node_id in ego_node_meta:
        if node_id not in G:
            G.add_node(node_id)

    if G.number_of_nodes() == 0:
        fig = go.Figure()
        fig.add_annotation(text="No graph data", xref="paper", yref="paper",
                           x=0.5, y=0.5, showarrow=False, font=dict(size=16, color="gray"))
        fig.update_layout(template="plotly_dark", height=500)
        return fig

    pos = nx.spring_layout(G, k=1.5, seed=42)

    # Edge traces
    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    traces = [go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(width=1, color="rgba(150,150,150,0.5)"),
        hoverinfo="none", showlegend=False,
    )]

    # Categorize nodes
    categories = {"center": [], "sar": [], "clean": []}
    for node in G.nodes():
        meta = ego_node_meta.get(node, {})
        if meta.get("is_center"):
            categories["center"].append(node)
        elif meta.get("is_sar"):
            categories["sar"].append(node)
        else:
            categories["clean"].append(node)

    cat_config = {
        "center": {"color": CLR["red"], "size": 22, "name": "Alert Node"},
        "sar": {"color": CLR["orange"], "size": 14, "name": "SAR Neighbor"},
        "clean": {"color": "#AED6F1", "size": 10, "name": "Clean Neighbor"},
    }
    for cat, nodes in categories.items():
        if not nodes:
            continue
        cfg = cat_config[cat]
        traces.append(go.Scatter(
            x=[pos[n][0] for n in nodes],
            y=[pos[n][1] for n in nodes],
            mode="markers+text",
            marker=dict(size=cfg["size"], color=cfg["color"],
                        line=dict(width=1, color="white")),
            text=nodes, textposition="top center", textfont=dict(size=8),
            name=cfg["name"],
            hovertext=[f"{ego_node_meta.get(n, {}).get('full_id', n)}<br>Score: {ego_node_meta.get(n, {}).get('score', '?')}" for n in nodes],
            hoverinfo="text",
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        template="plotly_dark",
        title=f"Ego-Graph ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)",
        showlegend=True, hovermode="closest",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(t=40, b=10, l=10, r=10),
        height=520,
    )
    return fig


@app.callback(
    [Output("top-alert-info", "children"),
     Output("chart-ego-graph", "figure"),
     Output("investigation-alert", "is_open")],
    [Input("interval-component", "n_intervals"),
     Input("tabs", "active_tab"),
     Input("btn-refresh-investigation", "n_clicks")],
    prevent_initial_call=True,
)
def update_investigation(n_intervals, active_tab, refresh_clicks):
    state = sim_state.get()

    if state["top_alert"] is None:
        empty = go.Figure()
        empty.add_annotation(text="Waiting for simulation to complete...",
                             xref="paper", yref="paper", x=0.5, y=0.5,
                             showarrow=False, font=dict(size=16, color="gray"))
        empty.update_layout(template="plotly_dark", height=520,
                            xaxis=dict(visible=False), yaxis=dict(visible=False))
        return (html.P("Run simulation first.", className="text-muted"), empty, True)

    ta = state["top_alert"]
    info = dbc.Card([
        dbc.CardBody([
            html.P([html.Strong("Node: "), ta["node_id"]], className="mb-1"),
            html.P([html.Strong("Score: "), f"{ta['score']:.4f}"], className="mb-1"),
            html.P([html.Strong("SAR: "), "Yes" if ta.get("is_sar") else "No"], className="mb-1"),
            html.P([html.Strong("Batch: "), str(ta.get("batch", "--"))], className="mb-1"),
            html.Hr(),
            html.P([html.Strong("Ego-graph: "), f"{state['ego_node_count']} nodes"], className="mb-0"),
        ])
    ], className="shadow-sm", style={"borderLeft": f"4px solid {CLR['red']}"})

    fig_ego = _build_ego_plotly(state["ego_edges"], state["ego_node_meta"])
    return (info, fig_ego, False)


print("Analysis + Investigation callbacks registered")


Analysis + Investigation callbacks registered


In [13]:
# ═══════════════════════════════════════════════════════════
#  Launch Dashboard
# ═══════════════════════════════════════════════════════════

print("Launching AML Simulation Dashboard on port 8052...")
print("Controls: START to begin simulation, STOP to halt, RESET to clear")
app.run(jupyter_mode="inline", port=8052)

Launching AML Simulation Dashboard on port 8052...
Controls: START to begin simulation, STOP to halt, RESET to clear


In [ ]:
# ══ GPU Cleanup ══
_stop_event.set()
if _sim_thread and _sim_thread.is_alive():
    _sim_thread.join(timeout=3.0)
_cleanup_gpu()
print(f"GPU freed: {torch.cuda.memory_allocated()/1e6:.1f} MB allocated")

GPU freed: 0.0 MB allocated


: 